# Overlap matching between two tiles

Two neighbouring SEM tiles see the same contacts in their overlap strip. Here they are paired
with `match_overlap`, and the difference between the two measurements of each contact is shown.

- Both tiles are placed at their **nominal** mask position (tile-local + tile centre from the CSV).
- `dx_nm, dy_nm` = tile B − tile A for the same physical contact. The mask error is the same
  in both images, so this difference is pure SEM placement/calibration error. Stitching will
  correct it later.

Pick two overlapping tiles (see `tiles_overview`) and run all cells.

In [ ]:
DATA_DIR = r"path\to\data"   # folder with the .jpg files, the metadata .csv and Contour/
TILE_A = 0                    # indices in read_tile_index order
TILE_B = 1
MAX_MATCH_NM = 50.0           # pair contacts closer than this; keep < half the pitch
ARROW_SCALE = 20              # arrows in the plot are drawn this many times longer

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd

from affine_ransac.features.contact import detect_contacts
from affine_ransac.geometry.frames import pixel_to_tile_nm
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.io.sem_image import load_sem_image
from affine_ransac.overlap import match_overlap, overlap_box

In [ ]:
def sem_contacts_mask_nm(tile):
    """Detected SEM contact centres of one tile, in mask nm (nominal placement)."""
    image = load_sem_image(tile.image_path)
    found = detect_contacts(image)
    local = pixel_to_tile_nm(found.centers, image.shape, tile.fov_x_nm / image.shape[1])
    return local + (tile.center_x_nm, tile.center_y_nm)


tiles = read_tile_index(DATA_DIR)
tile_a, tile_b = tiles[TILE_A], tiles[TILE_B]
points_a, points_b = sem_contacts_mask_nm(tile_a), sem_contacts_mask_nm(tile_b)
print(f"Tile A {tile_a.tile_id}: {len(points_a)} contacts; tile B {tile_b.tile_id}: {len(points_b)} contacts")

## Match the overlap

In [ ]:
box = overlap_box((tile_a.center_x_nm, tile_a.center_y_nm), (tile_a.fov_x_nm, tile_a.fov_y_nm),
                  (tile_b.center_x_nm, tile_b.center_y_nm), (tile_b.fov_x_nm, tile_b.fov_y_nm))
if box is None:
    raise ValueError("These two tiles do not overlap")
x_min, x_max, y_min, y_max = box
print(f"Overlap: x {x_min:.0f} .. {x_max:.0f} nm ({x_max - x_min:.0f} wide), "
      f"y {y_min:.0f} .. {y_max:.0f} nm ({y_max - y_min:.0f} high)")

ia, ib = match_overlap(points_a, points_b, box, MAX_MATCH_NM)
pairs = pd.DataFrame({
    "x_nm": points_a[ia, 0], "y_nm": points_a[ia, 1],
    "dx_nm": points_b[ib, 0] - points_a[ia, 0],
    "dy_nm": points_b[ib, 1] - points_a[ia, 1],
})
print(f"Matched contacts in the overlap: {len(pairs)}")
print(f"B - A: dx mean {pairs.dx_nm.mean():.2f} nm (std {pairs.dx_nm.std():.2f}), "
      f"dy mean {pairs.dy_nm.mean():.2f} nm (std {pairs.dy_nm.std():.2f})")
pairs

## The overlap strip

Tile A contacts `+`, tile B contacts `x`. Each arrow goes from A to B for the same contact,
drawn `ARROW_SCALE` times longer than the real difference.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.add_patch(plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, fill=False, color="gray", linestyle="--"))
ax.scatter(points_a[ia, 0], points_a[ia, 1], marker="+", color="tab:blue", label=f"A: {tile_a.tile_id}")
ax.scatter(points_b[ib, 0], points_b[ib, 1], marker="x", color="tab:red", label=f"B: {tile_b.tile_id}")
ax.quiver(pairs.x_nm, pairs.y_nm, pairs.dx_nm, pairs.dy_nm, angles="xy", scale_units="xy",
          scale=1 / ARROW_SCALE, color="black", width=0.003)
ax.set_xlim(x_min - 2 * MAX_MATCH_NM, x_max + 2 * MAX_MATCH_NM)
ax.set_ylim(y_min - 2 * MAX_MATCH_NM, y_max + 2 * MAX_MATCH_NM)
ax.set_aspect("equal")
ax.set_xlabel("mask x (nm)")
ax.set_ylabel("mask y (nm)")
ax.set_title(f"B - A differences, arrows x{ARROW_SCALE}")
ax.legend()
plt.show()